### Loading Libraries and Params

In [0]:
!pip install openpyxl

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import os, json, builtins
from datetime import date
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"
QUERY_OUTPUT_31 = f"{VOL31}/hcp_level_model/ops_pipeline/query_output"
TARGET_VARIABLE_FOLDER = f"{VOL31}/notebook_output/target_variable"
OUTPUT_PATH = f"{TARGET_VARIABLE_FOLDER}/target_variable"

# Target = Kerendia NBRx written by the HCP in the cohort's prediction month (full calendar month, claims offset 0)
TARGET_COL = "KERENDIA_NBRX"

# Writer flag = at least one Kerendia NBRx in the cohort's 12 lookback months (lookback_start_date .. end of lookback_end_date month)
WRITER_COL = "IS_EXISTING_WRITER"
NBRX_FEATURES_PATH = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features_v7_cohort_aligned/nbrx_features"   # cross-check only

# Stratified sample for feature selection: every HCP, N random training months + all validation months
SAMPLE_MONTHS_PER_HCP = 5
SAMPLE_TRAIN_COHORTS = (1, 24)      # months the N are picked from (training)
SAMPLE_TEST_COHORTS = (25, 26)      # kept in full (validation)
SAMPLE_SEED = 42
SAMPLE_LIST_NAME = f"all_hcps_{SAMPLE_MONTHS_PER_HCP}_train_months"
SAMPLE_LIST_PATH = f"{VOL31}/hcp_level_model/dev_pipeline/sample_lists/{SAMPLE_LIST_NAME}"

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()

# Prediction window = the full prediction_month (1st .. last day); the cohort file has no prediction start/end columns
df_cohort = (spark.createDataFrame(cohorts[['cohort_id','dataset','prediction_month','lookback_start_date','lookback_end_date']].astype(str))
       .withColumn("PREDICTION_START_DATE", F.to_date("prediction_month"))
       .withColumn("PREDICTION_END_DATE", F.last_day("PREDICTION_START_DATE"))
       .withColumn("LOOKBACK_START_M", F.to_date("lookback_start_date"))
       .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date"))
       .drop("prediction_month", "lookback_start_date", "lookback_end_date"))

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_hcp = hcp_universe.count()
print(f"Claims cohorts: {len(cohorts)} ({cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}), "
      f"lookback end {cohorts.lookback_end_date.iloc[0]}..{cohorts.lookback_end_date.iloc[-1]} | HCP universe: {n_hcp:,}")

In [0]:
# Prediction windows span c1..c28 prediction months; the writer flag also needs the 12 lookback months (from c1's
# lookback start), and NBRx needs 365 days of history before the earliest month used
coh_pd = df_cohort.select("PREDICTION_START_DATE", "PREDICTION_END_DATE", "LOOKBACK_START_M").toPandas()
data_start = str(coh_pd.PREDICTION_START_DATE.min())                                   # 2024-07-01 (c1)
data_end = str(coh_pd.PREDICTION_END_DATE.max())                                       # 2026-10-31 (c28)
data_start_writer = str(coh_pd.LOOKBACK_START_M.min())                                 # 2023-06-01 (c1 lookback start)
data_start_nbrx = str(pd.Timestamp(data_start_writer) - pd.Timedelta(days=365))[:10]   # 365-day NBRx lookback
print(f"data_start_nbrx={data_start_nbrx} | data_start_writer={data_start_writer} | data_start={data_start} | data_end={data_end}")

In [0]:
# NBA 3.1 universe flags (C1..C5 inclusion criteria); only PROVIDER_ID is used downstream
hcp_universe_flags = spark.read.parquet(HCP_UNIVERSE_PATH)
flag_cols = [c for c in hcp_universe_flags.columns if c != "PROVIDER_ID"]
hcp_universe_flags.agg(*[F.sum(c).alias(f"total_{c}") for c in flag_cols]).display()

In [0]:
# LOAD PAID KERENDIA CLAIMS (data_start_nbrx .. data_end, from the claims cohorts)
query = f'''
    SELECT DISTINCT A.PATIENT_ID, A.PROVIDER_ID, A.CLAIM_ID, A.SVC_DT, B.PRODUCT_GROUP 
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE CLAIM_TYPE = 'PD'
    AND SVC_DT >= '{data_start_nbrx}' 
    AND SVC_DT <= '{data_end}'
    AND PRODUCT_GROUP = 'KERENDIA'
'''

paid_claims = Get_Data_Snowflakes(query)
print('Shape :', paid_claims.count(), len(paid_claims.columns))

In [0]:
from pyspark.sql.functions import col, min as spark_min, lag, expr, when, row_number

# Step 1: Assign rank per patient-product-date to pick the first claim
dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP", "SVC_DT").orderBy("CLAIM_ID")

paid_claims = paid_claims.withColumn("CLAIM_RANK", row_number().over(dedup_window))

# Step 2: Define a window for NBRx calculation (per patient-product history)
window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")

# Compute the first-ever claim date per patient-product
paid_claims = paid_claims.withColumn("MIN_SVC_DT", spark_min("SVC_DT").over(window_spec))

# Compute the lagged service date (previous claim date for that product)
paid_claims = paid_claims.withColumn("LAG_SVC_DT", lag("SVC_DT").over(window_spec))

# Step 3: Calculate NBRX_FLAG only for the first claim per day
paid_claims_w_nbrx = paid_claims.withColumn(
    "NBRX_FLAG",
    when((col("MIN_SVC_DT") == col("SVC_DT")) & (col("CLAIM_RANK") == 1), 1)
    .when((col("LAG_SVC_DT").isNotNull()) & (col("LAG_SVC_DT") < expr("date_add(SVC_DT, -365)")) & (col("CLAIM_RANK") == 1), 1)
    .otherwise(0)
)

# Step 4: Drop intermediate columns
paid_claims_w_nbrx = paid_claims_w_nbrx.drop("MIN_SVC_DT", "LAG_SVC_DT", "CLAIM_RANK")
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
paid_claims_w_nbrx.display()

In [0]:
paid_claims_w_nbrx = paid_claims_w_nbrx.select(['CLAIM_ID','PROVIDER_ID','SVC_DT','NBRX_FLAG']).distinct()
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
# Writer = the HCP has at least one Kerendia NBRx in the cohort's 12 lookback months
# (LOOKBACK_START_M .. last day of the LOOKBACK_END_M month; the gap month and the prediction month are not used).
# Built here, before the claims are cut to data_start, so the c1..c13 lookbacks keep their 2023-2024 claims.
lb_win = df_cohort.select("cohort_id", "LOOKBACK_START_M", F.last_day("LOOKBACK_END_M").alias("LOOKBACK_LAST_DAY"))
df_writer = (
    paid_claims_w_nbrx.filter(col("NBRX_FLAG") == 1)
    .withColumn("SVC_DT", F.to_date("SVC_DT"))
    .join(F.broadcast(lb_win),
          (F.col("SVC_DT") >= F.col("LOOKBACK_START_M")) & (F.col("SVC_DT") <= F.col("LOOKBACK_LAST_DAY")))
    .select("PROVIDER_ID", "cohort_id").distinct()
    .withColumn(WRITER_COL, F.lit(True))
)
print('HCP x cohort rows with a lookback NBRx:', df_writer.count())

In [0]:
paid_claims_w_nbrx = paid_claims_w_nbrx.filter(col('SVC_DT')>= data_start)
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
# Step 1: Keep claims in each cohort's prediction month (PREDICTION_START_DATE .. PREDICTION_END_DATE, inclusive)
coh_win = df_cohort.select("cohort_id", "PREDICTION_START_DATE", "PREDICTION_END_DATE")
df_kerendia = (
    paid_claims_w_nbrx.withColumn("SVC_DT", F.to_date("SVC_DT"))
    .join(F.broadcast(coh_win),
          (F.col("SVC_DT") >= F.col("PREDICTION_START_DATE")) & (F.col("SVC_DT") <= F.col("PREDICTION_END_DATE")))
    .select("PROVIDER_ID", "CLAIM_ID", "SVC_DT", "NBRX_FLAG", "cohort_id")
)
print('Shape :', df_kerendia.count(), len(df_kerendia.columns))

# Step 2: Aggregate NBRx per HCP x cohort
df_kerendia_agg = (
    df_kerendia.groupBy("PROVIDER_ID", "cohort_id")
    .agg(F.sum("NBRX_FLAG").alias(TARGET_COL))
)
print('Shape :', df_kerendia_agg.count(), len(df_kerendia_agg.columns))

# Step 3: Every NBA 3.1 universe HCP x every claims cohort (c1..c28)
provider_cohort_grid = hcp_universe.crossJoin(F.broadcast(df_cohort))
print('Shape :', provider_cohort_grid.count(), len(provider_cohort_grid.columns))

# Step 4: Left join target and writer flag (missing = 0 NBRx / non-writer); keys aligned with the 3.1 HCP feature outputs
df_kerendia_final = (
    provider_cohort_grid
    .join(df_kerendia_agg, on=["PROVIDER_ID", "cohort_id"], how="left")
    .join(df_writer, on=["PROVIDER_ID", "cohort_id"], how="left")
    .fillna(0, subset=[TARGET_COL])
    .fillna(False, subset=[WRITER_COL])
    .withColumn("SVC_DT_M", F.col("LOOKBACK_END_M"))
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "dataset", "SVC_DT_M", "PREDICTION_START_DATE", "PREDICTION_END_DATE",
            "HCP_COHORT_ID", F.col(TARGET_COL).cast("int").alias(TARGET_COL), WRITER_COL, "PULL_DT")
)

print('Shape :', df_kerendia_final.count(), len(df_kerendia_final.columns))
df_kerendia_final.display()

In [0]:
# Grid completeness, key uniqueness, universe membership, and NBRx totals reconcile with the windowed claims
n_rows = df_kerendia_final.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert df_kerendia_final.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert df_kerendia_final.filter(F.col(TARGET_COL) < 0).count() == 0
in_univ = df_kerendia.join(hcp_universe, "PROVIDER_ID", "left_semi").agg(F.sum("NBRX_FLAG")).collect()[0][0] or 0
assert df_kerendia_final.agg(F.sum(TARGET_COL)).collect()[0][0] == in_univ
assert df_kerendia_final.filter(F.col(WRITER_COL).isNull()).count() == 0, "writer flag has empty values"

# Writer flag cross-check: same rule as the NBRx feature TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH > 0
# (empty = 0). Both use the same claims, so only rows touched by a Snowflake refresh between the two pulls may differ.
_feat = (spark.read.format("delta").load(NBRX_FEATURES_PATH)
         .select("HCP_COHORT_ID", (F.coalesce(F.col("TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH"), F.lit(0)) > 0).alias("_FEAT_WRITER")))
_cmp = df_kerendia_final.select("HCP_COHORT_ID", WRITER_COL).join(_feat, "HCP_COHORT_ID", "left")
_no_feat = _cmp.filter(F.col("_FEAT_WRITER").isNull()).count()
_diff = _cmp.filter(F.col("_FEAT_WRITER").isNotNull() & (F.col(WRITER_COL) != F.col("_FEAT_WRITER"))).count()
print(f"Writer flag vs NBRx feature: {_diff:,} rows differ ({_diff / n_rows:.4%}), {_no_feat:,} rows not in the feature table")
assert _diff <= 0.001 * n_rows, "writer flag differs from the NBRx feature on more than 0.1% of rows"

(df_kerendia_final.groupBy("cohort_id", "dataset", "PREDICTION_START_DATE")
   .agg(F.sum(TARGET_COL).alias("TOTAL_NBRX"),
        F.sum((F.col(TARGET_COL) > 0).cast("int")).alias("HCPS_WITH_NBRX"),
        F.sum(F.col(WRITER_COL).cast("int")).alias("WRITER_ROWS"),
        F.round(F.avg(F.col(WRITER_COL).cast("int")), 4).alias("WRITER_SHARE"),
        F.max(TARGET_COL).alias("MAX_NBRX"))
   .withColumn("_N", F.expr("int(substr(cohort_id, 2))")).orderBy("_N").drop("_N")
   .display())

In [0]:
# Save the output file (Delta) at OUTPUT_PATH from cell 3
os.makedirs(TARGET_VARIABLE_FOLDER, exist_ok=True)
(df_kerendia_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH))
tv = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {tv.count():,} rows, {len(tv.columns)} columns -> {OUTPUT_PATH}")

### Stratified sample for feature selection
Keeps **every HCP** but only `SAMPLE_MONTHS_PER_HCP` (5) of its 24 training months (c1-c24), plus all of its validation months (c25-c26).

* **Writer ratio kept:** each HCP's 5 months are split between writer and non-writer months in proportion to that HCP's own 24 months; the leftover fractions are rounded so the total writer share of the sample equals the share in all training rows.
* **Random but repeatable:** months inside each HCP's writer / non-writer group are picked by a hash with `SAMPLE_SEED`.
* **Weights:** `WEIGHT_GROUP` = 1 (all HCPs kept), `WEIGHT_MONTH` = 24 / 5 = 4.8 for training rows and 1 for validation rows, `WEIGHT` = their product.
* HCP `0` (missing prescriber ID) is left out, as in the combined physician data.

Written to `SAMPLE_LIST_PATH` in the same layout as the other sample lists (`SAMPLE_LIST` widget of `NBA_3_1_Combine_Physician_Data`).

In [0]:
tv = spark.read.format("delta").load(OUTPUT_PATH)
lo, hi = SAMPLE_TRAIN_COHORTS
t_lo, t_hi = SAMPLE_TEST_COHORTS
base = (tv.filter(F.col("PROVIDER_ID") != "0")
          .select("PROVIDER_ID", "HCP_COHORT_ID", F.col("cohort_id").alias("COHORT_ID"),
                  F.expr("int(substr(cohort_id, 2))").alias("COHORT_N"), WRITER_COL))
train = base.filter(F.col("COHORT_N").between(lo, hi))
n_train_months = hi - lo + 1
K = SAMPLE_MONTHS_PER_HCP

# Writer months per HCP: floor(K x HCP writer share), plus one for the HCPs with the largest leftover fractions
# until the sample's writer total equals K x HCPs x (writer share of all training rows)
full = train.agg(F.count("*").alias("n"), F.sum(F.col(WRITER_COL).cast("int")).alias("w")).first()
per_hcp = (train.groupBy("PROVIDER_ID")
           .agg(F.sum(F.col(WRITER_COL).cast("int")).alias("n_w"), F.count("*").alias("n_all"))
           .withColumn("q", F.lit(K) * F.col("n_w") / F.col("n_all"))
           .withColumn("q_floor", F.floor("q").cast("int"))
           .withColumn("q_frac", F.col("q") - F.col("q_floor")))
assert per_hcp.filter(F.col("n_all") != n_train_months).count() == 0, "some HCPs do not have every training month"
n_hcps = per_hcp.count()
target_writer_rows = int(builtins.round(n_hcps * K * full.w / full.n))   # builtins: the connection notebook's `import *` replaces round
extra = target_writer_rows - per_hcp.agg(F.sum("q_floor")).first()[0]
per_hcp = (per_hcp.withColumn("_r", F.row_number().over(Window.orderBy(F.desc("q_frac"), F.xxhash64("PROVIDER_ID", F.lit(SAMPLE_SEED)))))
                  .withColumn("K_WRITER", F.col("q_floor") + ((F.col("_r") <= extra) & (F.col("q_frac") > 0)).cast("int")))

train_pick = (train.join(per_hcp.select("PROVIDER_ID", "K_WRITER"), "PROVIDER_ID")
              .withColumn("_k", F.when(F.col(WRITER_COL), F.col("K_WRITER")).otherwise(F.lit(K) - F.col("K_WRITER")))
              .withColumn("_rk", F.row_number().over(Window.partitionBy("PROVIDER_ID", WRITER_COL)
                                                     .orderBy(F.xxhash64("HCP_COHORT_ID", F.lit(SAMPLE_SEED)))))
              .filter(F.col("_rk") <= F.col("_k"))
              .withColumn("SPLIT", F.lit("train"))
              .withColumn("WEIGHT_MONTH", F.lit(n_train_months / K)))
test_rows = (base.filter(F.col("COHORT_N").between(t_lo, t_hi))
             .withColumn("SPLIT", F.lit("test")).withColumn("WEIGHT_MONTH", F.lit(1.0)))

sample = (train_pick.select(*base.columns, "SPLIT", "WEIGHT_MONTH").unionByName(test_rows)
          .withColumn("SAMPLE_GROUP", F.lit("ALL"))
          .withColumn("WEIGHT_GROUP", F.lit(1.0))
          .withColumn("WEIGHT", F.col("WEIGHT_GROUP") * F.col("WEIGHT_MONTH"))
          .select("PROVIDER_ID", "HCP_COHORT_ID", "COHORT_ID", "COHORT_N", "SPLIT", "SAMPLE_GROUP",
                  WRITER_COL, "WEIGHT_GROUP", "WEIGHT_MONTH", "WEIGHT"))
sample.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SAMPLE_LIST_PATH)
print(f"Target writer rows in the training sample: {target_writer_rows:,} -> {SAMPLE_LIST_PATH}")

In [0]:
# Rows per HCP, unique keys, writer share equal to the full training rows, validation months complete
s = spark.read.format("delta").load(SAMPLE_LIST_PATH)
n_s = s.count()
assert s.select("HCP_COHORT_ID").distinct().count() == n_s, "duplicate HCP_COHORT_ID in the sample"
n_test_months = t_hi - t_lo + 1
assert n_s == n_hcps * (K + n_test_months), (n_s, n_hcps * (K + n_test_months))
per = s.groupBy("PROVIDER_ID", "SPLIT").count().groupBy("SPLIT").agg(F.min("count").alias("lo"), F.max("count").alias("hi")).collect()
assert {r.SPLIT: (r.lo, r.hi) for r in per} == {"train": (K, K), "test": (n_test_months, n_test_months)}, per

s_tr = s.filter("SPLIT = 'train'").agg(F.count("*").alias("n"), F.sum(F.col(WRITER_COL).cast("int")).alias("w")).first()
assert s_tr.w == target_writer_rows, (s_tr.w, target_writer_rows)
print(f"Sample: {n_s:,} rows = {n_hcps:,} HCPs x ({K} training + {n_test_months} validation months)")
print(f"Training writer share: sample {s_tr.w / s_tr.n:.3%} ({s_tr.n - s_tr.w:,} non-writer : {s_tr.w:,} writer = "
      f"{(s_tr.n - s_tr.w) / s_tr.w:.2f} : 1) | all training rows {full.w / full.n:.3%} = {(full.n - full.w) / full.w:.2f} : 1")
(s.groupBy("SPLIT", "COHORT_ID", "COHORT_N").agg(F.count("*").alias("rows"), F.sum(F.col(WRITER_COL).cast("int")).alias("writer_rows"))
  .orderBy("COHORT_N").display())

In [0]:
print("END OF NOTEBOOK")

In [0]:
df_output = spark.read.format("delta").load(OUTPUT_PATH)
df_output.display()

In [0]:
df_output.filter(F.col("PROVIDER_ID") == "7848062").display()